The contents of this notebook were created with assistance from Claude generative AI.

**Note:** the results of this analysis were **not used in the final report**.

# ModernBERT Context-Length Ablation — 5-fold CV + Gold

**What this is:** an ablation on the truncation window. We take the tuned ModernBERT config selected
by 5-fold grouped cross-validation (`saved_models/tuned_hyperparams.json`) and re-run the **identical
recipe** at **max_len 256** and **max_len 1024** — the only thing that changes is the context length.
Everything else is held fixed: learning rate, weight decay, warmup ratio, epochs, class-weighting,
batch size, the 5 grouped folds (`StratifiedGroupKFold` on `link_id`), and the population-weighted
gold set.

For each length we report the four headline metrics — **macro-F1 (4-class)**, **macro-F1 (3-class
on-topic)**, **balanced accuracy**, **macro PR-AUC** — on:
1. **5-fold grouped CV** — tuned config refit on each fold's train split, scored on its val split,
   reported as mean +/- std (sample std, ddof=1).
2. **Gold** — model refit on the **full training pool**, evaluated on the held-out human gold set,
   population-weighted.

The tuned **ModernBERT@512** numbers from the main notebook (`cv_meanstd.csv` / `gold_results.csv`)
are shown as the reference row, so the tables read as **256 vs 512 vs 1024**.

> **Kernel: `mads-m2-classifiers`.** Single-GPU. Each length trains 5 fold-models (CV) plus one
> full-pool model (gold) = 6 fine-tunes; two lengths ~= 12 fine-tunes. Expect roughly **30-60 min**
> (1024 is the slow one). If you hit CUDA OOM at 1024, drop `BS` to 8 in the config cell.

In [ ]:
import os, sys
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")           # single GPU, same as the main notebook
## SET TO 0 while using GPU 1 !!!!!!!!!!
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
P15 = os.getcwd()
if P15 not in sys.path: sys.path.insert(0, P15)
import numpy as np, pandas as pd, torch
import m5_common as M

# ── ablation config ───────────────────────────────────────────────────────────
LENGTHS   = [256, 1024]          # the ablation: re-run the tuned recipe at these context lengths
REF_LEN   = 512                  # already computed in the main notebook -> shown as the reference row
MODEL     = "ModernBERT"
MODEL_NAME = M.TF_SPECS[MODEL][0]            # "answerdotai/ModernBERT-base"
BS        = M.TF_SPECS[MODEL][2]             # 16 — held fixed across lengths so only max_len varies
LABELS    = M.LABELS
MET       = ["macro_f1", "macro_f1_3class", "balanced_accuracy", "macro_pr_auc"]

# tuned ModernBERT hyperparameters (selected by 5-fold CV @512) — reused verbatim, only max_len changes
hp = M.read_tuned_hp()[MODEL]
THP = {k: hp[k] for k in ("lr", "weight_decay", "warmup_ratio", "epochs")}
THP["class_weighted"] = hp.get("class_weighted", True)

F = M.load_frames(); df, gold = F["df"].reset_index(drop=True), F["gold_text"]
folds = M.fold_indices(df, M.CV_N_SPLITS)    # the SAME 5 grouped folds used for tuning
yg, wg = gold["label"], gold["w"].to_numpy()

print(f"training pool = {len(df)}   gold = {len(gold)}   folds = {M.CV_N_SPLITS}   batch_size = {BS}")
print(f"ablation lengths = {LENGTHS}   (reference = {MODEL}@{REF_LEN})")
print("tuned hyperparameters reused at every length:", THP)

training pool = 4830   gold = 301   folds = 5   batch_size = 16
ablation lengths = [256, 1024]   (reference = ModernBERT@512)
tuned hyperparameters reused at every length: {'lr': 4.991029159030355e-05, 'weight_decay': 0.01622890417738726, 'warmup_ratio': 0.038038895602179296, 'epochs': 4, 'class_weighted': True}


## Run the ablation — 5-fold CV + full-pool gold refit, per length

For each `max_len`: fine-tune the tuned ModernBERT config on each of the 5 fold train-splits and score
its val-split (CV), then refit once on the full pool and score the gold set. Per-fold scores are
aggregated to mean +/- std.

In [3]:
import time

def train_one(train_df, max_len):
    return M.train_tf(MODEL_NAME, max_len, BS, train_df,
                      lr=THP["lr"], weight_decay=THP["weight_decay"],
                      warmup_ratio=THP["warmup_ratio"], epochs=THP["epochs"],
                      class_weighted=THP["class_weighted"])

cv_fold_scores = {}      # length -> per-fold DataFrame
gold_rows      = []      # one dict per length

for L in LENGTHS:
    t0 = time.time()
    # ── 5-fold grouped CV ──
    rows = []
    for k, (tr, va) in enumerate(folds):
        mdl, tok = train_one(df.iloc[tr], L)
        proba = M.predict_tf(mdl, tok, df.iloc[va], L, return_proba=True)
        pred  = pd.Series(np.asarray(LABELS)[proba.argmax(1)], index=df.iloc[va].index)
        va_y  = df.iloc[va]["label"]
        mm = M.macro_metrics(va_y, pred); ap = M.ap_per_class(va_y, proba)
        rows.append({"macro_f1": mm["macro_f1"], "macro_f1_3class": mm["macro_f1_3class"],
                     "balanced_accuracy": mm["balanced_accuracy"], "macro_pr_auc": ap["macro_pr_auc"]})
        del mdl; torch.cuda.empty_cache()
        print(f"  @{L} fold {k+1}/{len(folds)}  macroF1[4]={rows[-1]['macro_f1']:.3f}", flush=True)
    cv_fold_scores[L] = pd.DataFrame(rows)

    # ── gold: refit on the full pool, evaluate population-weighted ──
    mdl, tok = train_one(df, L)
    proba = M.predict_tf(mdl, tok, gold, L, return_proba=True)
    pred  = np.asarray(LABELS)[proba.argmax(1)]
    mm = M.macro_metrics(yg, pred, sample_weight=wg); ap = M.ap_per_class(yg, proba, sample_weight=wg)
    gold_rows.append({"model": f"{MODEL}@{L}", "macroF1[4]": round(mm["macro_f1"], 3),
                      "macroF1[3]": round(mm["macro_f1_3class"], 3),
                      "bal_acc": round(mm["balanced_accuracy"], 3),
                      "macro_PR_AUC": round(ap["macro_pr_auc"], 3)})
    del mdl; torch.cuda.empty_cache()
    cv4 = cv_fold_scores[L]["macro_f1"]
    print(f"@{L}: done ({time.time()-t0:.0f}s)  CV macroF1[4] = {cv4.mean():.3f} +/- {cv4.std():.3f}  "
          f"gold macroF1[4] = {gold_rows[-1]['macroF1[4]']:.3f}\n", flush=True)

<home>\.venvs\mads-m2-classifiers\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 136/136 [00:00<00:00, 8000.92it/s]
[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  @256 fold 1/5  macroF1[4]=0.503


Loading weights: 100%|██████████| 136/136 [00:00<00:00, 8242.66it/s]
[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  @256 fold 2/5  macroF1[4]=0.528


Loading weights: 100%|██████████| 136/136 [00:00<00:00, 8771.32it/s]
[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

  @1024 fold 4/5  macroF1[4]=0.566


Loading weights: 100%|██████████| 136/136 [00:00<00:00, 9703.09it/s]
[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  @1024 fold 5/5  macroF1[4]=0.549


Loading weights: 100%|██████████| 136/136 [00:00<00:00, 9374.59it/s]
[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


@1024: done (1366s)  CV macroF1[4] = 0.543 +/- 0.021  gold macroF1[4] = 0.388



## Results — 256 vs 512 (ref) vs 1024

The reference `@512` row is read back from the main notebook's `cv_meanstd.csv` and `gold_results.csv`
(no recompute). Tables are written to `ablation_cv_meanstd.csv` and `ablation_gold_results.csv`.

In [4]:
PRETTY_CV = {"macro_f1": "CV macroF1[4]", "macro_f1_3class": "CV macroF1[3]",
             "balanced_accuracy": "CV bal_acc", "macro_pr_auc": "CV macro_PR_AUC"}

# ── 5-fold CV table (mean +/- std), with the @512 reference row spliced in ──
cv_rows = {}
ref_cv = pd.read_csv("cv_meanstd.csv", index_col=0)
if MODEL in ref_cv.index:
    cv_rows[f"{MODEL}@{REF_LEN} (ref)"] = ref_cv.loc[MODEL].to_dict()
for L in LENGTHS:
    fs = cv_fold_scores[L]
    cv_rows[f"{MODEL}@{L}"] = {PRETTY_CV[k]: f"{fs[k].mean():.3f} +/- {fs[k].std():.3f}" for k in MET}

order = ([f"{MODEL}@{LENGTHS[0]}"] +
         ([f"{MODEL}@{REF_LEN} (ref)"] if f"{MODEL}@{REF_LEN} (ref)" in cv_rows else []) +
         [f"{MODEL}@{L}" for L in LENGTHS[1:]])
cv_tbl = pd.DataFrame(cv_rows).T.reindex(order)[[PRETTY_CV[k] for k in MET]]
print("5-fold grouped CV — tuned ModernBERT recipe at each context length (mean +/- std):")
display(cv_tbl)
cv_tbl.to_csv("ablation_cv_meanstd.csv")

# ── gold table, with the tuned @512 reference row spliced in ──
gold_tbl = pd.DataFrame(gold_rows).set_index("model")
try:
    gref = pd.read_csv("gold_results.csv")
    r = gref[(gref["stage"] == "tuned") & (gref["model"] == MODEL)]
    if len(r):
        r = r.iloc[0]
        gold_tbl.loc[f"{MODEL}@{REF_LEN} (ref)"] = {c: r[c] for c in gold_tbl.columns}
except Exception as e:
    print("(no @512 gold reference:", e, ")")
gold_tbl = gold_tbl.reindex(order)
print("\nGold (population-weighted) — tuned ModernBERT recipe at each context length:")
display(gold_tbl)
gold_tbl.to_csv("ablation_gold_results.csv")
print("\nsaved -> ablation_cv_meanstd.csv, ablation_gold_results.csv")

5-fold grouped CV — tuned ModernBERT recipe at each context length (mean +/- std):


,CV macroF1[4],CV macroF1[3],CV bal_acc,CV macro_PR_AUC
ModernBERT@256,0.529 +/- 0.020,0.615 +/- 0.021,0.525 +/- 0.018,0.566 +/- 0.022
ModernBERT@512 (ref),0.542 ± 0.013,0.615 ± 0.023,0.538 ± 0.011,0.580 ± 0.014
ModernBERT@1024,0.543 +/- 0.021,0.620 +/- 0.026,0.539 +/- 0.020,0.587 +/- 0.021



Gold (population-weighted) — tuned ModernBERT recipe at each context length:


,macroF1[4],macroF1[3],bal_acc,macro_PR_AUC
model,,,,
ModernBERT@256,0.374,0.462,0.496,0.480
ModernBERT@512 (ref),0.308,0.408,0.426,0.505
ModernBERT@1024,0.388,0.443,0.518,0.512



saved -> ablation_cv_meanstd.csv, ablation_gold_results.csv
